# AASIST + AST: Audio Spoof Detection with Audio Spectrogram Transformer

This notebook trains the **AASIST-AST** hybrid model — a combination of the original AASIST (Audio Anti-Spoofing using Integrated Spectro-Temporal Graph Attention Networks) with an **Audio Spectrogram Transformer (AST)** encoder.

## Architecture Overview
- **AASIST Branch**: Raw waveform → SincConv → ResNet encoder → Spectral & Temporal GATs → Graph pooling
- **AST Branch**: Raw waveform → Mel Spectrogram → Patch Embedding → Transformer Encoder → CLS token
- **Fusion**: Concatenation of both branches → Final binary classifier

## Expected Performance
The baseline AASIST achieves **EER: 0.83%, min t-DCF: 0.0275** on ASVspoof2019 LA eval set.  
The AASIST-AST hybrid is expected to improve upon this by leveraging global spectro-temporal attention from the Transformer.

---
**Runtime**: Use **GPU** (T4 or A100 recommended). Go to `Runtime → Change runtime type → GPU`.

## Step 1: Check GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2: Clone Repository and Install Dependencies

In [ ]:
import os

# Clone the forked repository
if not os.path.exists('aasist'):
    !git clone https://github.com/ahmadSh96/aasist.git

%cd aasist

# Switch to the AST integration branch
!git checkout feature/ast-integration
!git pull origin feature/ast-integration

# Install dependencies
!pip install -q torchcontrib soundfile torchaudio

print('\n✅ Setup complete!')

## Step 3: Download ASVspoof 2019 LA Dataset

> **Note**: The dataset is ~10GB. This will take several minutes.
> If you already have the dataset, skip this cell and set `database_path` in the config accordingly.

In [ ]:
import os

if not os.path.exists('LA'):
    print('Downloading ASVspoof 2019 LA dataset (~10GB)...')
    !wget -q --show-progress https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip
    print('Extracting...')
    !unzip -q LA.zip
    !rm LA.zip
    print('✅ Dataset ready!')
else:
    print('✅ Dataset already exists, skipping download.')

## Step 4: Verify Model Architecture

Let's inspect the AASIST-AST model to confirm it loads correctly and count parameters.

In [ ]:
import sys
sys.path.insert(0, '.')

import json
import torch

# Load config
with open('config/AASIST_AST.conf', 'r') as f:
    config = json.load(f)

model_config = config['model_config']

# Load model
from models.AASIST_AST import Model
model = Model(model_config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model: AASIST-AST')
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Test forward pass
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
dummy_input = torch.randn(2, 64600).to(device)  # batch=2, 4 seconds at 16kHz
with torch.no_grad():
    features, output = model(dummy_input)

print(f'\nForward pass test:')
print(f'  Input shape:    {dummy_input.shape}')
print(f'  Features shape: {features.shape}')
print(f'  Output shape:   {output.shape}')
print('\n✅ Model loaded and forward pass successful!')

## Step 5: Train the AASIST-AST Model

In [ ]:
# Run training
# --config: path to the AASIST_AST configuration file
# --output_dir: directory to save checkpoints and logs
# --seed: random seed for reproducibility
!python main.py \
    --config config/AASIST_AST.conf \
    --output_dir exp_result \
    --seed 1234

## Step 6: Evaluate the Best Model

After training, evaluate the best checkpoint on the evaluation set.

In [ ]:
import glob
import os

# Find the best model checkpoint
best_models = glob.glob('exp_result/**/weights/best.pth', recursive=True)
if best_models:
    best_model_path = best_models[0]
    print(f'Found best model: {best_model_path}')
    
    # Update config to point to the best model
    import json
    with open('config/AASIST_AST.conf', 'r') as f:
        eval_config = json.load(f)
    
    eval_config['model_path'] = best_model_path
    
    with open('config/AASIST_AST_eval.conf', 'w') as f:
        json.dump(eval_config, f, indent=4)
    
    # Run evaluation
    !python main.py \
        --config config/AASIST_AST_eval.conf \
        --output_dir exp_result \
        --seed 1234 \
        --eval
else:
    print('No best model found. Make sure training completed successfully.')

## Step 7: Compare Results

Compare AASIST-AST results against the baseline AASIST.

In [ ]:
import glob
import re

# Read metric logs
metric_logs = glob.glob('exp_result/**/metric_log.txt', recursive=True)

print('=' * 60)
print('RESULTS COMPARISON')
print('=' * 60)
print(f'{"Model":<20} {"EER (%)":<15} {"min t-DCF":<15}')
print('-' * 60)
print(f'{"AASIST (baseline)":<20} {"0.83":<15} {"0.0275":<15}')

for log_file in metric_logs:
    with open(log_file, 'r') as f:
        content = f.read()
    # Extract EER and t-DCF from log
    eer_match = re.search(r'EER: ([0-9.]+)', content)
    tdcf_match = re.search(r'min t-DCF: ([0-9.]+)', content)
    if eer_match and tdcf_match:
        eer = float(eer_match.group(1))
        tdcf = float(tdcf_match.group(1))
        print(f'{"AASIST-AST":<20} {eer:<15.3f} {tdcf:<15.5f}')

print('=' * 60)

## Step 8: Save Results to Google Drive (Optional)

In [ ]:
# Uncomment to mount Google Drive and save results
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r exp_result /content/drive/MyDrive/AASIST_AST_Results
# print('✅ Results saved to Google Drive!')